# 07. Sorting and searching

## Learning goals
1. Prove four sorting algorithms correct with invariants.
2. Derive their running times.
3. Prove the n log n lower bound for comparison sorts.
4. Write binary search with a loop invariant and apply it to answers, not only arrays.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## Insertion sort
**Invariant.** Before step i, `a[:i]` holds the original first i items in sorted order.
Step i inserts `a[i]` into place, so the invariant holds for i+1. At the end the whole list is sorted.
**Cost.** Worst case (reverse order) is 1+2+...+(n-1) = n(n-1)/2 shifts, so O(n^2). Sorted input costs O(n). Stable.

In [ ]:
import random
from understanding_python.algorithms import insertion_sort, merge_sort, quick_sort, heap_sort, counting_sort, is_sorted
rng = random.Random(21)
cases = [[], [1], [2, 1], [3, 3, 3], list(range(20)), list(range(20, 0, -1))] + [[rng.randint(-50, 50) for _ in range(rng.randint(0, 60))] for _ in range(20)]
for f in (insertion_sort, merge_sort, quick_sort, heap_sort):
    for c in cases:
        assert f(c) == sorted(c), (f.__name__, c)
print("all sorters agree with sorted() on", len(cases), "cases")

## Merge sort
**Correctness.** By induction on length: halves sort correctly, and merging two sorted lists yields a sorted list because each step takes the smaller head.
**Cost.** T(n) = 2T(n/2) + cn. The recursion tree has log2 n levels and each level does cn work, so T(n) = O(n log n) in every case.
It uses O(n) extra space and is stable because ties take the left item first.

In [ ]:
pairs = [(1, "b"), (0, "x"), (1, "a"), (0, "y")]
by_key = sorted(pairs, key=lambda p: p[0])
assert by_key == [(0, "x"), (0, "y"), (1, "b"), (1, "a")]     # Timsort is stable too
from understanding_python.algorithms import count_comparisons
import math
n = 512
data = list(range(n)); random.Random(5).shuffle(data)
cmp = count_comparisons(merge_sort, data)
print("merge sort comparisons", cmp, "bound n*log2(n) =", int(n * math.log2(n)))
assert cmp <= n * math.log2(n)

## Quicksort
Partition around a pivot, then recurse on both sides. A random pivot splits evenly in expectation.
**Expected cost.** Two elements i<j are compared only if one is the first pivot chosen among the values between them, with probability 2/(j-i+1).
Summing over all pairs gives about 2 n ln n comparisons, so expected O(n log n). The worst case is O(n^2) with probability that vanishes quickly.

In [ ]:
n = 512
data = [rng.randint(0, 10**6) for _ in range(n)]
c = count_comparisons(quick_sort, data)
print("quicksort comparisons", c, "2 n ln n =", int(2 * n * math.log(n)))
assert c < 4 * n * math.log(n)
assert quick_sort(list(range(300))) == list(range(300))          # no blow-up on sorted input
assert quick_sort([7] * 300) == [7] * 300                        # equal keys stay linear-ish

## Heap sort and counting sort
Heap sort builds a max-heap in O(n) and then extracts n times at O(log n) each: O(n log n) worst case, in place, not stable.
Counting sort tallies values below k and rebuilds the list in O(n + k). It avoids comparisons.

In [ ]:
d = [rng.randrange(20) for _ in range(200)]
assert counting_sort(d, 20) == sorted(d) == heap_sort(d)
assert is_sorted(counting_sort(d, 20)) and not is_sorted([2, 1])

## Lower bound for comparison sorting
A comparison sort is a binary decision tree. It needs one leaf for each of the n! orderings.
A binary tree of height h has at most 2^h leaves, so 2^h >= n!, giving h >= log2(n!).
Since n! >= (n/2)^(n/2), h >= (n/2) log2(n/2) = Omega(n log n). No comparison sort beats this bound.

In [ ]:
for n in (4, 8, 16, 32):
    assert math.log2(math.factorial(n)) >= (n / 2) * math.log2(n / 2)
print("log2(8!) =", round(math.log2(math.factorial(8)), 2))

## Binary search
**Invariant.** The target, if present, lies in `a[lo:hi]`. Each step halves `hi - lo`, so the loop runs at most floor(log2 n) + 1 times: O(log n).
`lower_bound` returns the first index with `a[i] >= target`. The same loop solves "smallest x such that a monotone predicate holds".

In [ ]:
from understanding_python.algorithms import binary_search, lower_bound, search_on_answer
import bisect
a = sorted(rng.sample(range(1000), 100))
for t in range(0, 1000, 7):
    i = binary_search(a, t)
    assert (i == -1) == (t not in a) and (i == -1 or a[i] == t)
    assert lower_bound(a, t) == bisect.bisect_left(a, t)
assert binary_search([], 3) == -1
# smallest integer whose square is at least 1_000_000
assert search_on_answer(lambda x: x * x >= 1_000_000, 0, 10**6) == 1000
# fewest days to ship 10 loads with capacity 3 per day
assert search_on_answer(lambda days: days * 3 >= 10, 1, 100) == 4

## Benchmark: linear against binary search
Fixed seed, 100000 sorted items, 300 probes.

In [ ]:
from understanding_python.bench import best_time
big = sorted(rng.sample(range(10**7), 100_000))
probes = [rng.randrange(10**7) for _ in range(300)]
t_lin = best_time(lambda: [p in big for p in probes], 1)
t_bin = best_time(lambda: [binary_search(big, p) for p in probes], 3)
print(f"linear {t_lin:.4f}s binary {t_bin:.5f}s")
assert t_bin < t_lin
n = 2000
rand = [rng.random() for _ in range(n)]
t_ins = best_time(lambda: insertion_sort(rand), 1)
t_mrg = best_time(lambda: merge_sort(rand), 1)
print(f"n={n}: insertion {t_ins:.4f}s merge {t_mrg:.4f}s")
assert t_mrg < t_ins